# Explore image-parameter results

This notebook reads the summary CSV produced by `plot_image_params_results.py`; it does **not** open or process FITS files. Set `CSV_PATH` below to the existing `image_parameter_summary.csv`. Once FITS extraction has been done and the CSV is accessible, plotting is light enough to do locally without Slurm.

The notebook needs `pandas`, `matplotlib`, and `ipywidgets` (as well as `numpy`). If the controls do not render, enable/install the Jupyter widgets support in your notebook environment.

In [1]:
from pathlib import Path

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

# Update this to the summary CSV written by the FITS extraction run.
CSV_PATH = Path("/Users/gechen/soft_links/research_curtin/NIGHT_project/img_param_test/parameter_plots_test2_finer_grid/image_parameter_summary.csv")


In [2]:
if not CSV_PATH.is_file():
    raise FileNotFoundError(f"Summary CSV not found: {CSV_PATH.resolve()}")

results = pd.read_csv(CSV_PATH)
required_columns = {
    "obsnum", "robust", "minuv", "tukey", "rms_mjy_beam",
    "n_components_all", "n_components_flags0",
}
missing = required_columns - set(results.columns)
if missing:
    raise ValueError(f"CSV is missing required columns: {', '.join(sorted(missing))}")

for column in ("robust", "minuv", "tukey", "rms_mjy_beam",
               "n_components_all", "n_components_flags0"):
    results[column] = pd.to_numeric(results[column], errors="raise")
results["obsnum"] = results["obsnum"].astype(str)

print(f"Loaded {len(results):,} runs from {CSV_PATH.resolve()}")
results.head()


Loaded 7,401 runs from /Users/gechen/Dropbox/Mac3_Air2025/DB_Documents_Mac3/Research_Curtin/NIGHT_project/img_param_test/parameter_plots_test2_finer_grid/image_parameter_summary.csv


,obsnum,robust,minuv,tukey,rms_mjy_beam,valid_rms_fraction,n_components_all,n_components_flags0,rms_file,catalogue_file
0,1437423080,-0.1,0.0,0.0,3.536095,1.0,9589,9568,/scratch/pawsey0272/gcchen/test3_image_paramet...,/scratch/pawsey0272/gcchen/test3_image_paramet...
1,1437423080,-0.1,0.0,100.0,3.538592,1.0,9585,9564,/scratch/pawsey0272/gcchen/test3_image_paramet...,/scratch/pawsey0272/gcchen/test3_image_paramet...
2,1437423080,-0.1,0.0,200.0,3.540582,1.0,9588,9562,/scratch/pawsey0272/gcchen/test3_image_paramet...,/scratch/pawsey0272/gcchen/test3_image_paramet...
3,1437423080,-0.1,0.0,400.0,3.547289,1.0,9565,9543,/scratch/pawsey0272/gcchen/test3_image_paramet...,/scratch/pawsey0272/gcchen/test3_image_paramet...
4,1437423080,-0.1,0.0,50.0,3.537100,1.0,9589,9569,/scratch/pawsey0272/gcchen/test3_image_paramet...,/scratch/pawsey0272/gcchen/test3_image_paramet...


## Interactive plots

Choose one observation or all observations. Adjust the robust range and minimum-UV maximum to zoom the horizontal axes. These are display limits; the source CSV is not changed.

In [3]:
PARAMS = ("robust", "minuv", "tukey")
LABELS = {
    "robust": "Briggs robust",
    "minuv": "Minimum UV distance (wavelengths)",
    "tukey": "Inner Tukey taper (wavelengths)",
}
observations = sorted(results["obsnum"].unique())
observation = widgets.Dropdown(
    options=[("All observations", "All")] + [(obs, obs) for obs in observations],
    value="All",
    description="Observation:",
    layout=widgets.Layout(width="350px"),
)
count_metric = widgets.Dropdown(
    options=[("Components with flags = 0", "n_components_flags0"),
             ("All catalogue components", "n_components_all")],
    value="n_components_flags0",
    description="Counts:",
    layout=widgets.Layout(width="350px"),
)
robust_range = widgets.FloatRangeSlider(
    value=[float(results.robust.min()), float(results.robust.max())],
    min=float(results.robust.min()), max=float(results.robust.max()),
    step=max((float(results.robust.max()) - float(results.robust.min())) / 200, 0.001),
    description="Robust:", readout_format=".3g", continuous_update=False,
    layout=widgets.Layout(width="600px"),
)
minuv_max = widgets.FloatSlider(
    value=float(results.minuv.max()), min=float(results.minuv.min()),
    max=float(results.minuv.max()), step=max(float(results.minuv.max() - results.minuv.min()) / 200, 0.01),
    description="Minuv max:", readout_format=".3g", continuous_update=False,
    layout=widgets.Layout(width="600px"),
)

def plot_results(observation, count_metric, robust_range, minuv_max):
    selected = results if observation == "All" else results[results.obsnum == observation]
    if selected.empty:
        print("No rows match this selection.")
        return

    fig, axes = plt.subplots(2, 3, figsize=(15, 8), constrained_layout=True)
    for col, param in enumerate(PARAMS):
        other_params = [name for name in PARAMS if name != param]
        group_columns = other_params + ([] if observation != "All" else ["obsnum"])
        for axis, metric in zip(axes[:, col], ("rms_mjy_beam", count_metric)):
            for _, group in selected.groupby(group_columns, dropna=False):
                group = group.sort_values(param)
                axis.plot(group[param], group[metric], color="0.65",
                          linewidth=0.6, alpha=0.5)
            axis.scatter(selected[param], selected[metric], s=13,
                         alpha=0.5, color="tab:blue")
            medians = selected.groupby(param, as_index=False)[metric].median().sort_values(param)
            axis.plot(medians[param], medians[metric], "o-", color="tab:orange",
                      linewidth=2, label="Median over other settings")
            if param == "robust":
                axis.set_xlim(*robust_range)
            elif param == "minuv":
                axis.set_xlim(right=minuv_max)
            axis.set_xlabel(LABELS[param])
            axis.grid(alpha=0.2)
        axes[0, col].set_ylabel("Median map RMS (mJy/beam)")
        axes[1, col].set_ylabel(
            "Components (flags = 0)" if count_metric == "n_components_flags0"
            else "All catalogue components"
        )
    axes[0, 0].legend(fontsize=8)
    title = "All observations" if observation == "All" else f"Observation {observation}"
    fig.suptitle(f"{title} — {len(selected)} evaluated combinations\n"
                 "Points: individual runs; grey lines: other settings fixed.", fontsize=12)
    plt.show()

controls = widgets.VBox([
    widgets.HBox([observation, count_metric]),
    robust_range,
    minuv_max,
])
display(controls)
plot_output = widgets.interactive_output(
    plot_results,
    {"observation": observation, "count_metric": count_metric,
     "robust_range": robust_range, "minuv_max": minuv_max},
)
display(plot_output)


Output()

## Save a plot

To save the currently selected view, run the cell below. It writes a PNG into the current working directory. Change `PNG_PATH` if you want it somewhere else.

In [4]:
PNG_PATH = Path("image_parameter_exploration.png")

def save_current_view(path=PNG_PATH):
    selected = results if observation.value == "All" else results[results.obsnum == observation.value]
    if selected.empty:
        raise ValueError("No rows match the current observation selection.")
    fig, axes = plt.subplots(2, 3, figsize=(15, 8), constrained_layout=True)
    metric = count_metric.value
    for col, param in enumerate(PARAMS):
        other_params = [name for name in PARAMS if name != param]
        group_columns = other_params + ([] if observation.value != "All" else ["obsnum"])
        for axis, y_metric in zip(axes[:, col], ("rms_mjy_beam", metric)):
            for _, group in selected.groupby(group_columns, dropna=False):
                group = group.sort_values(param)
                axis.plot(group[param], group[y_metric], color="0.65", linewidth=0.6, alpha=0.5)
            axis.scatter(selected[param], selected[y_metric], s=13, alpha=0.5, color="tab:blue")
            medians = selected.groupby(param, as_index=False)[y_metric].median().sort_values(param)
            axis.plot(medians[param], medians[y_metric], "o-", color="tab:orange", linewidth=2)
            if param == "robust":
                axis.set_xlim(*robust_range.value)
            elif param == "minuv":
                axis.set_xlim(right=minuv_max.value)
            axis.set_xlabel(LABELS[param])
            axis.grid(alpha=0.2)
        axes[0, col].set_ylabel("Median map RMS (mJy/beam)")
        axes[1, col].set_ylabel("Components (flags = 0)" if metric == "n_components_flags0"
                                else "All catalogue components")
    axes[0, 0].legend(fontsize=8)
    title = "All observations" if observation.value == "All" else f"Observation {observation.value}"
    fig.suptitle(f"{title} — {len(selected)} evaluated combinations", fontsize=12)
    fig.savefig(path, dpi=160)
    plt.close(fig)
    print(f"Saved {path.resolve()}")

save_current_view()


/var/folders/l4/3h34w0390lsbgrl_sbf6915h0000gn/T/ipykernel_55283/396813067.py:28: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  axes[0, 0].legend(fontsize=8)


Saved /Users/gechen/Dropbox/Mac3_Air2025/DB_Documents_Mac3/Research_Curtin/NIGHT_project/night_light_code_laptop/image_parameter_exploration.png
